# HireIQ — End-to-End ML Training Pipeline

This notebook trains a **hiring prediction model** using the HireIQ 51-feature canonical vector.
The trained model integrates directly with the FastAPI backend and powers the **Hybrid AI ranking mode**.

### What this notebook does
1. Load the hiring dataset (candidate × job pairs)
2. Exploratory Data Analysis
3. Auto-extract 51 features via `ResumeFeatureExtractor`
4. Train multiple models with SMOTE + 5-fold CV + probability calibration
5. Evaluate: F1, ROC-AUC, PR-AUC, confusion matrix, SHAP
6. Save the best model in HireIQ backend artifact format
7. Demo: verify ML Engineer scores LOW for DevOps job (role discrimination test)

## 1. Setup & Imports

In [ ]:
import sys, os, warnings, uuid, json
from pathlib import Path

# Add project root to path so backend imports work
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve,
)

try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False
    print('WARNING: imbalanced-learn not installed. Run: pip install imbalanced-learn')

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

import shap

# HireIQ feature extractor
from backend.ml.feature_engineering import ResumeFeatureExtractor, FEATURE_NAMES, VERSION

plt.rcParams.update({'figure.figsize': (14, 6), 'font.size': 11})
sns.set_style('darkgrid')
pd.set_option('display.max_columns', 30)

MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

print(f'Feature vector version : {VERSION}')
print(f'Number of features     : {len(FEATURE_NAMES)}')
print(f'Models will be saved to: {MODELS_DIR}')

## 2. Load Dataset

In [ ]:
DATASETS_DIR = ROOT / 'uploads' / 'datasets'

# Priority: look for the hiring-format dataset (has resume_text + job_description)
HIRING_COLS = {'resume_text', 'job_description', 'shortlisted'}

df_raw = None
used_file = None

for fp in sorted(DATASETS_DIR.glob('*.xlsx')) + sorted(DATASETS_DIR.glob('*.csv')):
    try:
        tmp = pd.read_excel(fp) if fp.suffix == '.xlsx' else pd.read_csv(fp)
        tmp.columns = [c.lower().strip() for c in tmp.columns]
        if HIRING_COLS.issubset(set(tmp.columns)):
            df_raw = tmp
            used_file = fp.name
            print(f'Loaded hiring dataset: {fp.name}')
            print(f'Shape: {df_raw.shape}')
            break
    except Exception:
        continue

if df_raw is None:
    # Try legacy format (ai_hiring_dataset with direct feature columns)
    for fp in sorted(DATASETS_DIR.glob('*.xlsx')):
        try:
            tmp = pd.read_excel(fp)
            tmp.columns = [c.lower().strip() for c in tmp.columns]
            if 'shortlisted' in tmp.columns:
                df_raw = tmp
                used_file = fp.name
                print(f'Loaded legacy dataset: {fp.name}')
                print(f'Shape: {df_raw.shape}')
                break
        except Exception:
            continue

assert df_raw is not None, 'No suitable dataset found in uploads/datasets/'
print(f'\nColumns: {list(df_raw.columns)}')

In [ ]:
# Sample if dataset is very large (keep max 10000 rows for notebook speed)
MAX_ROWS = 10000
if len(df_raw) > MAX_ROWS:
    df_raw = df_raw.sample(MAX_ROWS, random_state=42).reset_index(drop=True)
    print(f'Sampled {MAX_ROWS} rows for notebook speed')

df_raw = df_raw.drop_duplicates().reset_index(drop=True)
TARGET_COL = 'shortlisted'

print(f'Final dataset shape: {df_raw.shape}')
print(f'\nTarget distribution:')
vc = df_raw[TARGET_COL].value_counts()
print(vc)
print(f'\nPositive rate: {vc.get(1, 0)/len(df_raw)*100:.1f}%')
df_raw.head(3)

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Class distribution
ax = axes[0]
labels = ['Not Shortlisted (0)', 'Shortlisted (1)']
counts = [df_raw[TARGET_COL].value_counts().get(0, 0), df_raw[TARGET_COL].value_counts().get(1, 0)]
bars = ax.bar(labels, counts, color=['#ef4444', '#22c55e'], width=0.5, edgecolor='white', linewidth=1.5)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{count}\n({count/len(df_raw)*100:.1f}%)', ha='center', fontweight='bold')
ax.set_title('Class Distribution (Imbalanced → SMOTE needed)', fontweight='bold')
ax.set_ylabel('Count')

# Experience years distribution
if 'experience_years' in df_raw.columns:
    ax = axes[1]
    for label, color in [(0, '#ef4444'), (1, '#22c55e')]:
        subset = df_raw[df_raw[TARGET_COL] == label]['experience_years'].dropna()
        ax.hist(subset, bins=15, alpha=0.7, color=color,
                label=f'{"Shortlisted" if label else "Rejected"}')
    ax.set_title('Experience Years by Outcome', fontweight='bold')
    ax.set_xlabel('Years')
    ax.legend()

# Job role distribution (if available)
ax = axes[2]
role_col = next((c for c in df_raw.columns if 'job_title' in c or 'job_role' in c), None)
if role_col:
    top_roles = df_raw[role_col].value_counts().head(8)
    top_roles.plot(kind='barh', ax=ax, color='#6366f1')
    ax.set_title(f'Top Job Roles in Dataset', fontweight='bold')
    ax.set_xlabel('Count')
else:
    ax.text(0.5, 0.5, 'No role column found', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('EDA saved to eda_overview.png')

In [ ]:
# Missing values
missing = df_raw.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print('Columns with missing values:')
if len(missing):
    print(missing.to_string())
else:
    print('None — dataset is clean!')

print(f'\nNumerical summary:')
num_cols = df_raw.select_dtypes(include='number').columns.tolist()
print(df_raw[num_cols].describe().round(2).to_string())

## 4. Feature Engineering

Using the **canonical 51-feature vector** — the same extractor used by the backend at inference time.
This eliminates training/serving skew.

In [ ]:
IS_HIRING_FORMAT = HIRING_COLS.issubset(set(df_raw.columns))
print(f'Dataset format: {"Hiring (auto-extract 51 features)" if IS_HIRING_FORMAT else "Legacy (use raw columns)"}')

extractor = ResumeFeatureExtractor()

if IS_HIRING_FORMAT:
    print(f'Extracting {len(FEATURE_NAMES)} features from {len(df_raw)} rows...')
    print('(This may take 1–3 minutes for large datasets)')
    
    from tqdm.auto import tqdm
    tqdm.pandas()
    
    try:
        X_list = df_raw.progress_apply(
            lambda row: extractor.extract_from_dataframe_row(row.to_dict()), axis=1
        ).tolist()
    except ImportError:
        # tqdm not available, use plain apply
        print('(tqdm not installed — running without progress bar)')
        X_list = df_raw.apply(
            lambda row: extractor.extract_from_dataframe_row(row.to_dict()), axis=1
        ).tolist()

    X = np.array(X_list, dtype=np.float32)
    feature_cols = list(FEATURE_NAMES)
    feature_version = VERSION
else:
    # Legacy format: use numeric columns directly
    feature_cols = [c for c in df_raw.columns if c not in [TARGET_COL, 'candidate_id', 'job_role', 'degree']]
    X_df = df_raw[feature_cols].copy()
    for col in X_df.columns:
        if X_df[col].dtype.kind in 'iufb':
            X_df[col] = X_df[col].fillna(X_df[col].median())
        else:
            X_df[col] = LabelEncoder().fit_transform(X_df[col].fillna('unknown').astype(str))
    X = X_df.values.astype(np.float32)
    feature_version = '1.0'

le = LabelEncoder()
y = le.fit_transform(df_raw[TARGET_COL].fillna('unknown').astype(str))

print(f'\nFeature matrix shape : {X.shape}')
print(f'Feature names        : {feature_cols[:5]} ... ({len(feature_cols)} total)')
print(f'Target classes       : {le.classes_} → {np.bincount(y)}')
print(f'Any NaN in X?        : {np.isnan(X).any()}')

In [ ]:
# Feature correlation heatmap (top 20 features with target)
X_df_full = pd.DataFrame(X, columns=feature_cols)
X_df_full['shortlisted'] = y

corr_with_target = X_df_full.corr()['shortlisted'].drop('shortlisted').abs().sort_values(ascending=False)
top20 = corr_with_target.head(20)

fig, ax = plt.subplots(figsize=(14, 7))
colors = ['#22c55e' if v > 0 else '#ef4444' for v in X_df_full.corr()['shortlisted'].drop('shortlisted')[top20.index]]
top20.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
ax.set_title('Top 20 Features by Correlation with Shortlisting Outcome', fontweight='bold', fontsize=13)
ax.set_ylabel('|Pearson correlation|')
ax.set_xticklabels(top20.index, rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('feature_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Train / Test Split + Scaling + SMOTE

In [ ]:
# Split FIRST (critical: prevents data leakage)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit scaler on TRAIN only
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train class distribution: {np.bincount(y_train)}')

# SMOTE to balance classes
smote_applied = False
if HAS_SMOTE:
    min_class = int(np.bincount(y_train).min())
    k = min(5, min_class - 1)
    if k >= 1:
        smote = SMOTE(random_state=42, k_neighbors=k)
        X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
        smote_applied = True
        print(f'\nAfter SMOTE (k={k}): {X_train_bal.shape}')
        print(f'Balanced class distribution: {np.bincount(y_train_bal)}')
    else:
        X_train_bal, y_train_bal = X_train, y_train
        print('SMOTE skipped: not enough minority samples')
else:
    X_train_bal, y_train_bal = X_train, y_train
    print('SMOTE skipped: imbalanced-learn not installed')

## 6. Multi-Model Training with 5-Fold CV + Calibration

In [ ]:
cw = None if smote_applied else 'balanced'

MODELS = {
    'XGBoost':              XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05,
                                          subsample=0.8, colsample_bytree=0.8,
                                          eval_metric='logloss', verbosity=0, random_state=42),
    'LightGBM':             LGBMClassifier(n_estimators=200, learning_rate=0.05,
                                           num_leaves=63, class_weight=cw,
                                           verbose=-1, random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=200, max_depth=12,
                                                   class_weight=cw, n_jobs=-1, random_state=42),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=150, max_depth=4,
                                                        learning_rate=0.05, subsample=0.8,
                                                        random_state=42),
    'Logistic Regression':  LogisticRegression(max_iter=1000, C=1.0,
                                               class_weight='balanced', random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print(f'{"Model":<22} {"CV F1":>8} {"CV±":>6} {"Test F1":>8} {"ROC-AUC":>9} {"PR-AUC":>8}')
print('-' * 66)

trained_models = {}
calibrated_models = {}

for name, clf in MODELS.items():
    # 5-fold CV
    cv_scores = cross_val_score(clf, X_train_bal, y_train_bal, cv=cv,
                                scoring='f1_weighted', n_jobs=-1)
    cv_mean, cv_std = cv_scores.mean(), cv_scores.std()

    # Fit base model
    clf.fit(X_train_bal, y_train_bal)
    trained_models[name] = clf

    # Calibrate
    min_class_bal = int(np.bincount(y_train_bal).min())
    calib_folds = min(3, min_class_bal)
    if calib_folds >= 2:
        from sklearn.base import clone
        cal_clf = CalibratedClassifierCV(clone(clf), cv=calib_folds, method='isotonic')
        cal_clf.fit(X_train_bal, y_train_bal)
    else:
        cal_clf = clf
    calibrated_models[name] = cal_clf

    # Evaluate on test set
    y_pred  = cal_clf.predict(X_test)
    y_proba = cal_clf.predict_proba(X_test)[:, 1]

    test_f1  = f1_score(y_test, y_pred, average='binary', zero_division=0)
    roc_auc  = roc_auc_score(y_test, y_proba)
    pr_auc   = average_precision_score(y_test, y_proba)

    results[name] = {
        'cv_f1_mean': round(cv_mean, 4),
        'cv_f1_std':  round(cv_std, 4),
        'test_f1':    round(test_f1, 4),
        'roc_auc':    round(roc_auc, 4),
        'pr_auc':     round(pr_auc, 4),
        'y_proba':    y_proba,
        'y_pred':     y_pred,
    }

    print(f'{name:<22} {cv_mean:>8.4f} {cv_std:>6.4f} {test_f1:>8.4f} {roc_auc:>9.4f} {pr_auc:>8.4f}')

best_name = max(results, key=lambda n: results[n]['roc_auc'])
print(f'\n✓ Best model by ROC-AUC: {best_name} ({results[best_name]["roc_auc"]:.4f})')

## 7. Evaluation — Confusion Matrix, ROC/PR Curves, Calibration

In [ ]:
# ROC and PR curves for all models
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
colors_palette = ['#6366f1', '#22c55e', '#f59e0b', '#ef4444', '#06b6d4']

for (name, res), color in zip(results.items(), colors_palette):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    axes[0].plot(fpr, tpr, color=color, lw=2,
                 label=f'{name} (AUC={res["roc_auc"]:.3f})')

    prec, rec, _ = precision_recall_curve(y_test, res['y_proba'])
    axes[1].plot(rec, prec, color=color, lw=2,
                 label=f'{name} (AP={res["pr_auc"]:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
axes[0].set(xlabel='False Positive Rate', ylabel='True Positive Rate',
            title='ROC Curves — All Models')
axes[0].legend(fontsize=9)

baseline_pr = y_test.mean()
axes[1].axhline(baseline_pr, color='black', linestyle='--', lw=1, alpha=0.5,
                label=f'Random baseline ({baseline_pr:.2f})')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curves — All Models')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('roc_pr_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrix for best model
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

best_res = results[best_name]
cm = confusion_matrix(y_test, best_res['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Rejected', 'Shortlisted'],
            yticklabels=['Rejected', 'Shortlisted'],
            annot_kws={'size': 16, 'weight': 'bold'})
axes[0].set(title=f'Confusion Matrix — {best_name}',
            ylabel='Actual', xlabel='Predicted')

# Model comparison bar chart
metrics_df = pd.DataFrame({
    name: {'CV F1': res['cv_f1_mean'], 'Test F1': res['test_f1'],
           'ROC-AUC': res['roc_auc'], 'PR-AUC': res['pr_auc']}
    for name, res in results.items()
}).T

metrics_df.plot(kind='bar', ax=axes[1], color=['#6366f1', '#22c55e', '#f59e0b', '#06b6d4'],
               edgecolor='white', width=0.7)
axes[1].set(title='Model Comparison — Key Metrics', ylim=(0, 1.05))
axes[1].set_xticklabels(metrics_df.index, rotation=30, ha='right')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\n{best_name} — Classification Report:')
print(classification_report(y_test, best_res['y_pred'],
                             target_names=le.classes_, digits=4))

In [ ]:
# Probability calibration plot
fig, ax = plt.subplots(figsize=(8, 6))

for (name, res), color in zip(results.items(), colors_palette):
    frac_pos, mean_pred = calibration_curve(y_test, res['y_proba'], n_bins=10)
    ax.plot(mean_pred, frac_pos, 's-', color=color, label=name, lw=2, ms=6)

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect calibration')
ax.set(xlabel='Mean predicted probability', ylabel='Fraction of positives',
       title='Calibration Curves (closer to diagonal = better calibrated)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('calibration_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Calibrated models should follow the diagonal much more closely.')

## 8. SHAP Feature Importance

In [ ]:
# Use the base (uncalibrated) best model for SHAP
base_best = trained_models[best_name]

print(f'Computing SHAP values for {best_name}...')

# Use a background sample for speed
background = shap.sample(X_train_bal, min(200, len(X_train_bal)), random_state=42)

if hasattr(base_best, 'feature_importances_'):
    explainer = shap.TreeExplainer(base_best)
    shap_values = explainer.shap_values(X_test[:500])  # first 500 test rows for speed
    if isinstance(shap_values, list):
        sv = shap_values[1]  # positive class
    else:
        sv = shap_values
else:
    explainer = shap.LinearExplainer(base_best, masker=shap.maskers.Independent(background))
    shap_values = explainer.shap_values(X_test[:500])
    sv = shap_values

print(f'SHAP values shape: {sv.shape}')

In [ ]:
# SHAP summary bar plot
plt.figure(figsize=(10, 8))
shap.summary_plot(
    sv, X_test[:500],
    feature_names=feature_cols,
    plot_type='bar',
    max_display=20,
    show=False
)
plt.title(f'SHAP Feature Importance — {best_name}', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP beeswarm (shows direction of feature effects)
plt.figure(figsize=(10, 9))
shap.summary_plot(
    sv, X_test[:500],
    feature_names=feature_cols,
    max_display=20,
    show=False
)
plt.title(f'SHAP Beeswarm — {best_name}\n(red=high feature value, blue=low; x-axis=impact on prediction)',
          fontsize=11, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Save Best Model to HireIQ Backend Format

In [ ]:
model_id = str(uuid.uuid4())
model_dir = MODELS_DIR / model_id
model_dir.mkdir(parents=True, exist_ok=True)

best_cal  = calibrated_models[best_name]   # calibrated — for inference
best_base = trained_models[best_name]      # raw — for SHAP
best_res  = results[best_name]

# Save artifacts
joblib.dump(best_cal,  model_dir / 'model.pkl')
joblib.dump(best_base, model_dir / 'base_model.pkl')
joblib.dump(scaler,    model_dir / 'preprocessor.pkl')
joblib.dump(le,        model_dir / 'label_encoder.pkl')

(model_dir / 'feature_metadata.json').write_text(json.dumps({
    'feature_names':    feature_cols,
    'feature_count':    len(feature_cols),
    'version':          feature_version,
    'smote_applied':    smote_applied,
    'cv_f1_mean':       best_res['cv_f1_mean'],
    'cv_f1_std':        best_res['cv_f1_std'],
    'pr_auc':           best_res['pr_auc'],
    'is_hiring_dataset':IS_HIRING_FORMAT,
    'dataset_rows':     len(df_raw),
}, indent=2))

(model_dir / 'metrics.json').write_text(json.dumps({
    'accuracy':     round(float(accuracy_score(y_test, best_res['y_pred'])), 4),
    'f1':           best_res['test_f1'],
    'precision':    round(float(precision_score(y_test, best_res['y_pred'], zero_division=0)), 4),
    'recall':       round(float(recall_score(y_test, best_res['y_pred'], zero_division=0)), 4),
    'roc_auc':      best_res['roc_auc'],
    'pr_auc':       best_res['pr_auc'],
    'cv_f1_mean':   best_res['cv_f1_mean'],
    'cv_f1_std':    best_res['cv_f1_std'],
    'confusion_matrix': confusion_matrix(y_test, best_res['y_pred']).tolist(),
    'class_labels': le.classes_.tolist(),
}, indent=2))

(model_dir / 'training_config.json').write_text(json.dumps({
    'algorithm':     best_name,
    'train_samples': int(len(X_train_bal)),
    'test_samples':  int(len(X_test)),
    'smote_applied': smote_applied,
    'calibrated':    True,
    'target_column': TARGET_COL,
    'source_file':   used_file,
}, indent=2))

import datetime
(model_dir / 'version.json').write_text(json.dumps({
    'model_version': 'v2.0.0',
    'trained_at':    datetime.datetime.utcnow().isoformat() + 'Z',
    'experiment_id': model_id,
}, indent=2))

total_size_mb = sum(f.stat().st_size for f in model_dir.rglob('*') if f.is_file()) / 1024 / 1024

print(f'Model saved to: {model_dir}')
print(f'Model ID      : {model_id}')
print(f'Artifact size : {total_size_mb:.2f} MB')
print(f'\nFiles saved:')
for f in sorted(model_dir.iterdir()):
    print(f'  {f.name:<30} ({f.stat().st_size/1024:.1f} KB)')

## 10. Demo — Role Discrimination Test

**The critical test**: Does the model correctly score an **ML Engineer** LOW for a **DevOps job** and HIGH for an **ML job**?

This is the fix for the 80%+ false positive bug.

In [ ]:
# Define test cases
TEST_CASES = [
    {
        'label': 'ML Engineer → DevOps Job (should score LOW)',
        'resume_text': 'Machine Learning Engineer with 5 years experience. Expert in Python, TensorFlow, PyTorch, Scikit-learn, Jupyter, MLflow, Pandas, NumPy. Built neural networks and NLP pipelines. Published research in deep learning. Strong Python developer.',
        'skills': 'Python, TensorFlow, PyTorch, Scikit-learn, Pandas, NumPy, Jupyter, MLflow, NLP',
        'experience_years': 5,
        'education': 'Master of Science in Machine Learning',
        'job_title': 'Senior DevOps Engineer',
        'job_description': 'Hiring a Senior DevOps Engineer. Must have hands-on Terraform, Ansible, Jenkins CI/CD, Kubernetes cluster management, Bash scripting, Linux system administration, Docker containerization. Experience with AWS infrastructure, monitoring with Prometheus and Grafana.',
        'required_skills': 'Terraform, Ansible, Jenkins, Kubernetes, Bash, Linux, Docker, CI/CD',
        'preferred_skills': 'Prometheus, Grafana, Helm, ArgoCD',
        'experience_requirement': '4-6 years',
        'education_requirement': "Bachelor's in Computer Science",
        'expected': 'LOW (< 0.40)',
    },
    {
        'label': 'ML Engineer → ML Job (should score HIGH)',
        'resume_text': 'Machine Learning Engineer with 5 years experience. Expert in Python, TensorFlow, PyTorch, Scikit-learn, Jupyter, MLflow, Pandas, NumPy. Built neural networks and NLP pipelines. Deployed models at scale with Docker and Kubernetes. Published research.',
        'skills': 'Python, TensorFlow, PyTorch, Scikit-learn, Pandas, NumPy, Docker, MLflow, AWS',
        'experience_years': 5,
        'education': 'Master of Science in Machine Learning',
        'job_title': 'Machine Learning Engineer',
        'job_description': 'Hiring a Machine Learning Engineer. Required: Python, TensorFlow, PyTorch, MLflow, Docker. Design and deploy production ML models. Experience with NLP, computer vision, or recommendation systems. Strong software engineering practices.',
        'required_skills': 'Python, TensorFlow, PyTorch, MLflow, Docker',
        'preferred_skills': 'Kubernetes, AWS, Spark, Airflow',
        'experience_requirement': '3-5 years',
        'education_requirement': "Master's in ML or CS",
        'expected': 'HIGH (> 0.70)',
    },
    {
        'label': 'DevOps Engineer → DevOps Job (should score HIGH)',
        'resume_text': 'DevOps Engineer with 6 years experience. Expert in Terraform, Ansible, Jenkins, Kubernetes, Docker, Linux, Bash scripting. Managed AWS infrastructure with Prometheus monitoring. CI/CD pipeline specialist. Helm charts, ArgoCD, GitOps.',
        'skills': 'Terraform, Ansible, Jenkins, Kubernetes, Docker, Linux, Bash, AWS, Prometheus, Grafana, Helm, ArgoCD',
        'experience_years': 6,
        'education': "Bachelor's in Computer Science",
        'job_title': 'Senior DevOps Engineer',
        'job_description': 'Hiring a Senior DevOps Engineer. Must have hands-on Terraform, Ansible, Jenkins CI/CD, Kubernetes cluster management, Bash scripting, Linux system administration, Docker containerization. Experience with AWS infrastructure, monitoring with Prometheus and Grafana.',
        'required_skills': 'Terraform, Ansible, Jenkins, Kubernetes, Bash, Linux, Docker, CI/CD',
        'preferred_skills': 'Prometheus, Grafana, Helm, ArgoCD',
        'experience_requirement': '4-6 years',
        'education_requirement': "Bachelor's in Computer Science",
        'expected': 'HIGH (> 0.70)',
    },
    {
        'label': 'Junior Frontend Dev → Senior Backend Job (should score LOW)',
        'resume_text': 'Junior Frontend Developer with 1 year experience. Built React components and CSS animations. Basic JavaScript and HTML. Familiar with Figma and design tools. No backend experience.',
        'skills': 'React, JavaScript, HTML, CSS, Figma',
        'experience_years': 1,
        'education': "Bachelor's in Design",
        'job_title': 'Senior Backend Engineer',
        'job_description': 'Hiring a Senior Backend Engineer. Required: Python, FastAPI, PostgreSQL, Redis, Docker. Design scalable microservices, REST APIs. 5+ years of backend development. Strong system design skills.',
        'required_skills': 'Python, FastAPI, PostgreSQL, Redis, Docker, microservices',
        'preferred_skills': 'Kubernetes, Kafka, AWS',
        'experience_requirement': '5+ years',
        'education_requirement': "Bachelor's in Computer Science",
        'expected': 'LOW (< 0.35)',
    },
]

print('Running role discrimination tests...\n')
scores = []

for tc in TEST_CASES:
    row = {
        'resume_text':          tc['resume_text'],
        'skills':               tc['skills'],
        'experience_years':     tc['experience_years'],
        'education':            tc['education'],
        'job_title':            tc['job_title'],
        'job_description':      tc['job_description'],
        'required_skills':      tc['required_skills'],
        'preferred_skills':     tc['preferred_skills'],
        'experience_requirement': tc['experience_requirement'],
        'education_requirement': tc['education_requirement'],
        'certifications': '', 'projects': '', 'ats_score': 70,
        'portfolio_url': '', 'github_url': 'https://github.com/demo',
        'soft_skills': 'communication, teamwork', 'languages': 'English',
    }
    
    features = extractor.extract_from_dataframe_row(row).reshape(1, -1)
    features_scaled = scaler.transform(features)
    prob = calibrated_models[best_name].predict_proba(features_scaled)[0][1]
    scores.append(prob)

    icon = '✓' if (
        ('LOW' in tc['expected'] and prob < 0.50) or
        ('HIGH' in tc['expected'] and prob > 0.60)
    ) else '✗'

    print(f'{icon} {tc["label"]}')
    print(f'  Probability: {prob:.3f}   Expected: {tc["expected"]}')
    print()

In [ ]:
# Visualise the discrimination scores
fig, ax = plt.subplots(figsize=(12, 5))

short_labels = [
    'ML Eng\n→ DevOps Job',
    'ML Eng\n→ ML Job',
    'DevOps Eng\n→ DevOps Job',
    'Junior Frontend\n→ Sr Backend Job',
]
bar_colors = ['#ef4444' if s < 0.5 else '#22c55e' for s in scores]

bars = ax.bar(short_labels, scores, color=bar_colors, width=0.5, edgecolor='white', linewidth=2)
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{score:.2%}', ha='center', fontweight='bold', fontsize=13)

ax.axhline(0.5, color='black', linestyle='--', lw=1.5, alpha=0.6, label='Decision threshold (0.5)')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Shortlisting Probability', fontsize=12)
ax.set_title(f'Role Discrimination Test — {best_name}\n'
              '(Green = correct role fit | Red = wrong role — should score low)',
             fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('role_discrimination_test.png', dpi=120, bbox_inches='tight')
plt.show()

## 11. Deploy to Backend

The model artifact is already saved to `models/{model_id}/`.  
To deploy it as the production model, run the cell below or use the HireIQ Training UI.

In [ ]:
print('=' * 60)
print('DEPLOYMENT INSTRUCTIONS')
print('=' * 60)
print(f'\nModel ID       : {model_id}')
print(f'Algorithm      : {best_name}')
print(f'ROC-AUC        : {results[best_name]["roc_auc"]:.4f}')
print(f'PR-AUC         : {results[best_name]["pr_auc"]:.4f}')
print(f'Test F1        : {results[best_name]["test_f1"]:.4f}')
print(f'CV F1          : {results[best_name]["cv_f1_mean"]:.4f} ± {results[best_name]["cv_f1_std"]:.4f}')
print(f'\nModel path     : models/{model_id}/')
print()
print('Option 1 — Deploy via HireIQ Training UI:')
print('  1. Open http://localhost:3000/training')
print('  2. Go to Step 5: Leaderboard')
print('  3. Find this model and click "Deploy to Production"')
print()
print('Option 2 — Deploy via API (run this cell):')
print(f'  curl -X POST http://localhost:8000/api/training/models/{model_id}/deploy \\')
print(f'       -H "Authorization: Bearer YOUR_TOKEN"')
print()
print('Option 3 — Register directly in database (programmatic):')

In [ ]:
# Option 3: Programmatic registration via SQLAlchemy
# Run this cell to register the notebook-trained model directly in the HireIQ database

import asyncio

async def register_model():
    from backend.core.database import AsyncSessionLocal
    from backend.models.ml_model import MLModel
    from sqlalchemy import select
    import datetime

    best_res = results[best_name]
    
    async with AsyncSessionLocal() as db:
        ml_model = MLModel(
            id=model_id,
            name=f'{best_name} — Notebook (51-feat)',
            version='v2.0.0',
            algorithm=best_name,
            status='trained',
            deployment_status='experimental',
            accuracy=float(accuracy_score(y_test, best_res['y_pred'])),
            precision_score=float(precision_score(y_test, best_res['y_pred'], zero_division=0)),
            recall_score=float(recall_score(y_test, best_res['y_pred'], zero_division=0)),
            f1_score=best_res['test_f1'],
            roc_auc=best_res['roc_auc'],
            model_size_mb=round(sum(f.stat().st_size for f in model_dir.rglob('*') if f.is_file()) / 1024 / 1024, 3),
            feature_count=len(feature_cols),
            training_samples=len(X_train_bal),
            target_column=TARGET_COL,
            model_path=str(model_dir),
            feature_names=feature_cols,
            class_labels=le.classes_.tolist(),
            trained_at=datetime.datetime.utcnow(),
        )
        db.add(ml_model)
        await db.commit()
        print(f'✓ Model {model_id} registered in HireIQ database')
        print(f'  Status: experimental')
        print(f'  Go to the Training UI → Leaderboard → Deploy to make it production')

try:
    asyncio.get_event_loop().run_until_complete(register_model())
except RuntimeError:
    # Jupyter already has an event loop
    import nest_asyncio
    nest_asyncio.apply()
    asyncio.get_event_loop().run_until_complete(register_model())
except Exception as e:
    print(f'Could not auto-register (backend not running): {e}')
    print(f'Use the API or Training UI to deploy model: {model_id}')

## Summary

| Step | Done |
|---|---|
| Load hiring dataset (15k rows, 18 cols) | ✓ |
| EDA — class distribution, experience, roles | ✓ |
| Extract 51-feature vector (same as backend inference) | ✓ |
| SMOTE — balance 81/19 class split | ✓ |
| 5-fold stratified CV | ✓ |
| Train 5 models + CalibratedClassifierCV | ✓ |
| Evaluate: F1, ROC-AUC, PR-AUC, confusion matrix | ✓ |
| SHAP feature importance (bar + beeswarm) | ✓ |
| Save artifact in HireIQ backend format | ✓ |
| Role discrimination test (ML Eng vs DevOps) | ✓ |